In this notebook, we implement a way to collect all the parameters (basically weights and bias) of internal nodes.
Because we cannot touch weights and bias of input nodes on the peripherals of the network.

And then try to tweak them, to reduce the loss. And help NN learn to predict what we want.

In [1]:
from graphviz import Digraph
import math

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [2]:
class Value:
    def __init__(self, data, _children=(), _op = '', label = ''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0.0
        self.label = label
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data = {self.data})"

    def __add__(self, other):
        # This line is added later
        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): # other + self
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')
        def _backward():
            # _backward is basically derivative of self**other
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        # This line is added later
        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    # This is something not needed, it is a python trick to be able to do
    # a = Value(3.0)
    # 2 * a --> This can be done because of __rmul__
    def __rmul__(self, other):
        return self * other

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # We have to set this or else it will be 0.
        self.grad = 1.0
        
        for node in reversed(topo):
            node._backward()


In [3]:

import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)] 
        self.b = Value(random.uniform(-1, 1)) 

    def __call__(self, x):
        assert len(x) == len(self.w), f"Dimension mismatch: Neuron expects {len(self.w)} inputs, got {len(x)}"
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        # Implemented new in this notebook.
        # This mimics pytorch's implementation.
        # This is basically convenient way to gather all parameters.
        return self.w + [self.b]

class Layer: 
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [4]:
xs = [
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5], 
    [0.5, 1.0, 1.0], 
    [1.0, 1.0, -1.0],
]
n = MLP(3, [4, 4, 1])
ys = [1.0, -1.0, -1.0, 1.0] # desired targets 


In [5]:
ypred = [n(x)[0] for x in xs]
loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
loss

Value(data = 6.727109493340318)

In [6]:
loss.backward()

In [7]:
# This is the gradient of the neuron
print(n.layers[0].neurons[0].w[0].grad)
# This is the current value of the neuron
print(n.layers[0].neurons[0].w[0].data)

-0.6380404925138785
-0.4241063808470138


In [8]:
# Total parameters = Layer 1 (4 neurons * 3 inputs + 4 biases) 
#                  + Layer 2 (4 neurons * 4 inputs + 4 biases) 
#                  + Layer 3 (1 neuron  * 4 inputs + 1 bias)
#                  = 16 + 20 + 5 = 41
print("number of parameters for this NN: ", len(n.parameters()))

number of parameters for this NN:  41


In [9]:
# We want to iterate over each parameter and try to nudge each parameter in the right direction.
# So that we can nudge the output or reduce the loss.
for p in n.parameters():
    # We update .data and not weight or bias. Because see __call__ function above in Neuron
    # __call__ function mixes weight, bias to produce data. It uses all available operations.
    # zip, *, then sum (or +), then tanh
    # So, nudging p.data basically means, nudging weight, bias.
    p.data += -0.01 * p.grad

# We don't want to maximize the loss, we want to decrease it. 
# I can think of the gradient vector—basically just the vector of all the gradients—as pointing in the direction of increasing the loss.
# But then we want to decrease it, so we actually want to go in the opposite direction.

In [10]:
# After nudging, the value is a tiny amount greater.
# Previously it was -0.4241063808470138
print(n.layers[0].neurons[0].w[0].data)

# Now, if we run forward pass. We will use these new data.

-0.417725975921875


In [11]:
ypred = [n(x)[0] for x in xs]
loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
loss

Value(data = 6.501650563898031)

as we can see the loss as reduced from 6.7something to 6.5

And we can continue this process of 

loss.backward for new gradients
step size readjust p.data (basically weights and bias)
then forward pass to calculate loss

In [12]:
loss.backward() # We get new gradients

# Nudge or step size
for p in n.parameters():
    p.data += -0.01 * p.grad

# Forward pass
ypred = [n(x)[0] for x in xs]
# Calculate the loss
loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
print('loss: ', loss)



loss:  Value(data = 6.0168643945293745)


In [13]:
loss.backward() # We get new gradients

# Nudge or step size
for p in n.parameters():
    p.data += -0.01 * p.grad

# Forward pass
ypred = [n(x)[0] for x in xs]
# Calculate the loss
loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
print('loss: ', loss)



loss:  Value(data = 5.257746336403356)


In [14]:
loss.backward() # We get new gradients

# Nudge or step size
for p in n.parameters():
    p.data += -0.01 * p.grad

# Forward pass
ypred = [n(x)[0] for x in xs]
# Calculate the loss
loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
print('loss: ', loss)



loss:  Value(data = 4.454305263509548)


In [15]:
loss.backward() # We get new gradients

# Nudge or step size
for p in n.parameters():
    p.data += -0.01 * p.grad

# Forward pass
ypred = [n(x)[0] for x in xs]
# Calculate the loss
loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
print('loss: ', loss)



loss:  Value(data = 3.9874941654411584)


In [16]:
ypred

[Value(data = 0.9090121965561136),
 Value(data = -0.6021272499132407),
 Value(data = 0.9370770429793781),
 Value(data = 0.7379977302268499)]

In [19]:
for i in range(100):
    loss.backward() # We get new gradients
    
    # Nudge or step size
    for p in n.parameters():
        p.data += -0.01 * p.grad
    
    # Forward pass
    ypred = [n(x)[0] for x in xs]
    # Calculate the loss
    loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))
    if i % 10 == 0:
        print('loss: ', loss)

loss:  Value(data = 3.7880733931777404)
loss:  Value(data = 0.016238271379631044)
loss:  Value(data = 0.07528652892035487)
loss:  Value(data = 2.290443556538734e-06)
loss:  Value(data = 2.4272801565455012e-09)
loss:  Value(data = 8.567220836949317e-12)
loss:  Value(data = 7.231354204058422e-14)
loss:  Value(data = 1.4288108555996389e-15)
loss:  Value(data = 6.20806453590069e-17)
loss:  Value(data = 5.467034327245649e-18)


In [20]:
ypred

[Value(data = 0.9999999999981849),
 Value(data = -0.9999999999999954),
 Value(data = -0.9999999989778483),
 Value(data = 0.9999999999974036)]

# Wohoooo!

In [22]:
# This is the setting of weights/biases which is needed for our NN to output correctly.
n.parameters()

[Value(data = -2.7221312781737113),
 Value(data = -10.464371487350608),
 Value(data = 17.956521389082326),
 Value(data = -1.5955929268415827),
 Value(data = -3.6053968598761563),
 Value(data = -4.931998397682252),
 Value(data = 6.749003487064148),
 Value(data = -4.006633398312985),
 Value(data = 1.2530540761701268),
 Value(data = -9.81454216681299),
 Value(data = 8.651178345387347),
 Value(data = -3.44292978946382),
 Value(data = -0.37223059211799103),
 Value(data = -0.07785338259399868),
 Value(data = -2.518103177970543),
 Value(data = -0.2565543170824417),
 Value(data = -2.962394851471778),
 Value(data = 0.29856952365816714),
 Value(data = -1.0183868040400113),
 Value(data = -0.9440890140238017),
 Value(data = 1.4558916205679897),
 Value(data = -0.4246844853221662),
 Value(data = 3.9891865809885387),
 Value(data = 2.481166744919095),
 Value(data = 1.6710173389301481),
 Value(data = -1.0743561690218668),
 Value(data = 9.149076515183916),
 Value(data = 3.313166715544532),
 Value(data =